# RAG completo con Elasticsearch y LangChain 1.x

Este notebook practica un flujo completo de **Retrieval Augmented Generation (RAG)** usando un PDF sobre redes neuronales como fuente documental.

La práctica está dividida en tres partes:

1. **Ingesta de datos:** carga de PDF, fragmentación con `RecursiveCharacterTextSplitter`, embeddings y almacenamiento en Elasticsearch.
2. **Agente básico con RAG:** creación de un agente en LangChain 1.x usando una herramienta propia que envuelve el `vector_store` y permite filtros por metadata.
3. **Búsquedas avanzadas:** filtros por metadata, filtros por varios campos, BM25, búsqueda híbrida BM25 + vector, búsqueda híbrida + metadata, reranking y respuestas con sustento de documento y página.

> Importante: el objetivo didáctico es mostrar cómo la metadata permite pasar de una búsqueda semántica simple a una recuperación más controlada, explicable y auditable.

# Instalación de dependencias


In [4]:
%pip install -q "langchain>=1.0" langchain-google-genai langchain-elasticsearch langchain-community langchain-text-splitters pypdf elasticsearch pandas

# 1. Configuración de credenciales y variables


- `/content/api_key.txt` para `GEMINI_API_KEY`
- `/content/elasticstore_urp.txt` para la contraseña de Elasticsearch

También puedes usar variables de entorno.

In [37]:
import os
import getpass
from pathlib import Path


def read_secret_file(path: str) -> str | None:
    file_path = Path(path)
    if file_path.exists():
        return file_path.read_text(encoding="utf-8").strip()
    return None

# Gemini
gemini_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or read_secret_file("/content/api_key.txt")
if not gemini_key:
    gemini_key = getpass.getpass("Ingresa tu GEMINI_API_KEY: ")
os.environ["GEMINI_API_KEY"] = gemini_key
os.environ["GOOGLE_API_KEY"] = gemini_key

# Elasticsearch
ES_URL = os.getenv("ES_URL", "http://104.198.172.31:9200")
ES_USER = os.getenv("ES_USER", "elastic")
ES_PASSWORD = os.getenv("ES_PASSWORD") or read_secret_file("/content/elasticstore_urp.txt")
if not ES_PASSWORD:
    ES_PASSWORD = getpass.getpass("Ingresa la contraseña de Elasticsearch: ")

INDEX_NAME = os.getenv("ES_INDEX", "rag_hiraoka_terminosycondiciones")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "gemini-embedding-2")
CHAT_MODEL = os.getenv("CHAT_MODEL", "google_genai:gemini-2.5-flash")

print("Configuración cargada")
print("ES_URL:", ES_URL)
print("INDEX_NAME:", INDEX_NAME)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)
print("CHAT_MODEL:", CHAT_MODEL)

Configuración cargada
ES_URL: http://104.198.172.31:9200
INDEX_NAME: rag_hiraoka_terminosycondiciones
EMBEDDING_MODEL: gemini-embedding-2
CHAT_MODEL: google_genai:gemini-2.5-flash


# Parte 1 — Ingesta de datos

En esta parte cargaremos el PDF, construiremos metadata útil para búsquedas filtradas y guardaremos los chunks en Elasticsearch.

La metadata se construye considerando que el PDF trata sobre **redes neuronales**, con secciones como introducción, reseña histórica, generalidades, elementos básicos, aprendizaje, topologías, aplicaciones y software comercial.

## 1.1 Cargar el PDF en Colab

El notebook espera el archivo `matich-redesneuronales.pdf`. Si no existe en `/content`, se abrirá un uploader para cargarlo manualmente.

In [38]:
PDF_PATH = Path("/content/Terminos_Condiciones.pdf")

if not PDF_PATH.exists():
    try:
        from google.colab import files
        print("Sube el archivo Terminos_Condiciones.pdf")
        uploaded = files.upload()
        if uploaded:
            first_name = next(iter(uploaded.keys()))
            Path(first_name).rename(PDF_PATH)
            print(f"PDF cargado como: {PDF_PATH}")
    except Exception as e:
        raise FileNotFoundError(
            "No se encontró /content/Terminos_Condiciones.pdf. "
            "Sube el PDF manualmente o cambia PDF_PATH."
        ) from e

print("PDF listo:", PDF_PATH)

PDF listo: /content/Terminos_Condiciones.pdf


## 1.2 Loader para PDF

Usaremos `PyPDFLoader` para extraer el texto por página. Cada página se convierte en un `Document` de LangChain.

In [39]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print(f"Páginas cargadas: {len(pages)}")
print("Metadata base:", pages[0].metadata)

Páginas cargadas: 18
Metadata base: {'producer': 'Skia/PDF m148', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36 Edg/148.0.0.0', 'creationdate': '2026-06-10T14:54:11+00:00', 'title': '▷ Términos y Condiciones de compra en Hiraoka.com.pe', 'moddate': '2026-06-10T14:54:11+00:00', 'source': '/content/Terminos_Condiciones.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}


## 1.3 Enriquecimiento de metadata

La metadata permitirá filtrar por:

- `documento`
- `page_number`
- `section_slug`
- `section_title`
- `temas`
- `tipo_contenido`
- `dominio`
- `autor`
- `anio`

Esto es importante porque luego la herramienta del agente podrá buscar por tema, página, sección o tipo de contenido.

### Paso 1 - Crear secciones reales del documento

In [40]:
import re
import unicodedata
from dataclasses import dataclass
from typing import Optional

DOCUMENT_TITLE = "Terminos y Condiciones"
DOCUMENT_AUTHOR = "Hiraoka"
DOCUMENT_YEAR = 2026
DOCUMENT_DOMAIN = "Términos y Condiciones"


def normalize_text(text: str) -> str:
    return (
        unicodedata.normalize("NFKD", text.lower())
        .encode("ascii", "ignore")
        .decode("ascii")
    )


@dataclass
class Section:
    slug: str
    title: str
    parent: Optional[str]
    page_start: int
    page_end: int
    text: str


SECTION_TITLES = [
    # Principales
    {"slug": "uso_portal", "title": "Uso del Portal"},
    {"slug": "registro_usuario", "title": "Registro del Usuario"},
    {"slug": "proteccion_menores", "title": "Restricciones sobre algunos productos y protección de menores de edad"},
    {"slug": "clave_secreta", "title": "Clave Secreta"},
    {"slug": "derechos_usuario", "title": "Derechos del Usuario en el Sitio Web"},
    {"slug": "aceptacion_compra", "title": "Aceptación en los Procedimientos de Compra del Sitio Web"},

    # Procedimiento de pago y sus subsecciones
    {"slug": "procedimiento_pago", "title": "Procedimiento de Selección de Productos y Pago"},
    {"slug": "medios_pago", "title": "Medios de pago", "parent": "procedimiento_pago"},
    {"slug": "seguridad_pago", "title": "Medidas de seguridad adoptadas en el procedimiento de pago", "parent": "procedimiento_pago"},
    {"slug": "pago_izipay", "title": "Izipay", "parent": "seguridad_pago"},
    {"slug": "pago_efectivo", "title": "PagoEfectivo", "parent": "seguridad_pago"},
    {"slug": "pago_oka", "title": "Oka", "parent": "seguridad_pago"},

    {"slug": "imagenes_productos", "title": "Sobre las imágenes de los Productos"},
    {"slug": "precio_stock", "title": "Precio y Stock contenidos en el Sitio Web"},
    {"slug": "arma_combo", "title": "Arma tu combo"},

    # Entrega de productos y sus subsecciones
    {"slug": "entrega_productos", "title": "Mecanismos de entrega de productos"},
    {"slug": "recojo_tienda", "title": "Recojo en Tienda", "parent": "entrega_productos"},
    {"slug": "horario_atencion", "title": "los horarios de atención", "parent": "entrega_productos"},
    {"slug": "delivery", "title": "Servicio de envío o Delivery", "parent": "entrega_productos"},
    {"slug": "delivery_tamano", "title": "Tamaño", "parent": "delivery"},
    {"slug": "delivery_tipo", "title": "Tipo de servicio de envío (Delivery) elegido", "parent": "delivery"},
    {"slug": "delivery_costos", "title": "Tabla de costos", "parent": "delivery"},
    {"slug": "delivery_equivalencias", "title": "Cálculo de equivalencias de los costos de envío del Delivery", "parent": "delivery"},
    {"slug": "delivery_lima_puntual", "title": "Envíos puntuales dentro de Lima", "parent": "delivery"},

    {"slug": "reprogramaciones", "title": "Reprogramaciones"},
    {"slug": "propiedad_intelectual", "title": "Propiedad Intelectual e Industrial"},
    {"slug": "responsabilidad", "title": "Limitación de Responsabilidad"},
    {"slug": "enlaces", "title": "Enlaces (Links)"},
    {"slug": "fuerza_mayor", "title": "Fuerza Mayor"},
    {"slug": "contacto", "title": "¿Cómo contactar a Hiraoka?"},
    {"slug": "garantias", "title": "Garantías"},
    {"slug": "generalidades", "title": "Generalidades"}
]

### Paso 2 - Encontrar dónde empieza cada sección

In [41]:
def find_section_starts(pages):
    full_text = ""
    char_to_page = []

    for idx, page in enumerate(pages):
        start = len(full_text)
        full_text += page.page_content + "\n\n"
        end = len(full_text)
        char_to_page.append((start, end, idx + 1))

    def get_page_number(char_idx):
        for start, end, p_num in char_to_page:
            if start <= char_idx < end:
                return p_num
        return len(pages)

    norm_full = normalize_text(full_text)
    detected = []

    for sec in SECTION_TITLES:
        norm_title = normalize_text(sec["title"])
        pos = norm_full.find(norm_title)
        if pos != -1:
            detected.append({
                "slug": sec["slug"],
                "title": sec["title"],
                "parent": sec.get("parent"),
                "char_start": pos,
                "page": get_page_number(pos)
            })

    detected.sort(key=lambda x: x["char_start"])
    return detected, full_text, get_page_number

### Paso 3 - Construir objetos Section

In [42]:
def build_sections(pages):
    starts, full_text, get_page_number = find_section_starts(pages)
    sections = []

    for i, section in enumerate(starts):
        start_pos = section["char_start"]
        end_pos = starts[i+1]["char_start"] if i < len(starts)-1 else len(full_text)

        # Extraer el texto exacto entre esta sección y la siguiente
        section_text = full_text[start_pos:end_pos].strip()
        page_start = section["page"]
        page_end = get_page_number(end_pos - 1)

        sections.append(
            Section(
                slug=section["slug"],
                title=section["title"],
                parent=section["parent"],
                page_start=page_start,
                page_end=page_end,
                text=section_text
            )
        )
    return sections

### Ejecutar

In [43]:
sections = build_sections(pages)

print(f"Secciones encontradas: {len(sections)}")

for s in sections:
    print(
        s.slug,
        s.page_start,
        s.page_end
    )

Secciones encontradas: 32
pago_oka 1 1
uso_portal 1 1
fuerza_mayor 1 2
propiedad_intelectual 2 2
registro_usuario 2 2
clave_secreta 2 3
proteccion_menores 3 3
derechos_usuario 3 3
aceptacion_compra 3 4
procedimiento_pago 4 5
garantias 5 5
medios_pago 5 5
pago_izipay 5 5
pago_efectivo 5 5
seguridad_pago 5 7
imagenes_productos 7 7
precio_stock 7 8
arma_combo 8 8
entrega_productos 8 8
recojo_tienda 8 8
delivery_tamano 8 9
horario_atencion 9 10
delivery 10 11
delivery_tipo 11 13
delivery_costos 13 13
delivery_equivalencias 13 15
delivery_lima_puntual 15 15
reprogramaciones 15 16
contacto 16 16
responsabilidad 16 16
enlaces 16 17
generalidades 17 18


## 1.4 Fragmentación con RecursiveCharacterTextSplitter

Fragmentaremos por página para preservar la trazabilidad. Cada chunk hereda la metadata de su página original.

In [45]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

### Convertimos las secciones a documentos:

In [46]:
section_docs = []

# Mapeamos los títulos de secciones por su slug para resolver la jerarquía
sections_by_slug = {s["slug"]: s for s in SECTION_TITLES}

for section in sections:
    metadata = {
        "documento": DOCUMENT_TITLE,
        "autor": DOCUMENT_AUTHOR,
        "anio": DOCUMENT_YEAR,
        "dominio": DOCUMENT_DOMAIN,
        "page_start": section.page_start,
        "page_end": section.page_end
    }

    # Determinar jerarquía
    parent_slug = section.parent
    if not parent_slug:
        # Es sección principal (Main)
        metadata["section_slug"] = section.slug
        metadata["section_title"] = section.title
    else:
        grandparent_slug = sections_by_slug[parent_slug].get("parent")
        if not grandparent_slug:
            # Es subsección (Sub)
            metadata["section_slug"] = parent_slug
            metadata["section_title"] = sections_by_slug[parent_slug]["title"]
            metadata["subsection_slug"] = section.slug
            metadata["subsection_title"] = section.title
        else:
            # Es sub-subsección (Sub-sub)
            metadata["section_slug"] = grandparent_slug
            metadata["section_title"] = sections_by_slug[grandparent_slug]["title"]
            metadata["subsection_slug"] = parent_slug
            metadata["subsection_title"] = sections_by_slug[parent_slug]["title"]
            metadata["sub_subsection_slug"] = section.slug
            metadata["sub_subsection_title"] = section.title

    doc = Document(
        page_content=section.text,
        metadata=metadata
    )
    section_docs.append(doc)

In [47]:
splits = splitter.split_documents(
    section_docs
)

In [48]:
print(splits[0].page_content[:900])

www.hiraoka.com.pe/ (en
adelante, el “Sitio Web”) es IMPORTACIONES HIRAOKA SA.C. (en adelante, “Hiraoka”), con RUC N°
201000016681, domiciliado en Avenida Abancay N° 594, distrito Cercado de Lima, provincia y departamento de


### Paso 5 - Metadata a nivel chunk

In [50]:
for idx, chunk in enumerate(splits):
    specific_slug = (
        chunk.metadata.get("sub_subsection_slug")
        or chunk.metadata.get("subsection_slug")
        or chunk.metadata.get("section_slug")
    )
    chunk.metadata["chunk_id"] = f"{specific_slug}_{idx:04d}"
    chunk.metadata["chunk_number"] = idx

In [17]:
print(splits[2].metadata)

{'documento': 'Terminos y Condiciones', 'autor': 'Hiraoka', 'anio': 2026, 'dominio': 'Términos y Condiciones', 'page_start': 1, 'page_end': 1, 'section_slug': 'uso_portal', 'section_title': 'Uso del Portal', 'chunk_id': 'uso_portal_0002', 'chunk_number': 2}


## 1.5 Resumen de metadata generada

Antes de indexar, revisamos la distribución de chunks por sección y tipo de contenido.

In [51]:
import pandas as pd

metadata_df = pd.DataFrame([doc.metadata for doc in splits])

resumen_secciones = (
    metadata_df.groupby(["section_title", "page_start"])
    .size()
    .reset_index(name="chunks")
    .sort_values("chunks", ascending=False)
)

resumen_tipo = (
    metadata_df.groupby("section_title")
    .size()
    .reset_index(name="chunks")
    .sort_values("chunks", ascending=False)
)

print("Chunks por sección")
display(resumen_secciones)
print("Chunks por tipo de contenido")
display(resumen_tipo)

Chunks por sección


,section_title,page_start,chunks
18,Procedimiento de Selección de Productos y Pago,5,16
11,Mecanismos de entrega de productos,10,13
0,Aceptación en los Procedimientos de Compra del...,3,8
4,Enlaces (Links),16,7
9,Mecanismos de entrega de productos,8,6
15,Precio y Stock contenidos en el Sitio Web,7,5
13,Mecanismos de entrega de productos,13,5
22,Restricciones sobre algunos productos y protec...,3,5
24,Uso del Portal,1,5
8,Limitación de Responsabilidad,16,4


Chunks por tipo de contenido


,section_title,chunks
9,Mecanismos de entrega de productos,33
11,Procedimiento de Selección de Productos y Pago,19
0,Aceptación en los Procedimientos de Compra del...,8
4,Enlaces (Links),7
17,Uso del Portal,5
15,Restricciones sobre algunos productos y protec...,5
10,Precio y Stock contenidos en el Sitio Web,5
8,Limitación de Responsabilidad,4
2,Clave Secreta,4
5,Fuerza Mayor,4


## 1.6 Embeddings y conexión a Elasticsearch

Conectamos a `ElasticsearchStore` y guardaremos los chunks con IDs estables basados en documento, página y número de chunk.

In [52]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = ElasticsearchStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    es_url=ES_URL,
    es_user=ES_USER,
    es_password=ES_PASSWORD,
)

# Cambia a False si quieres reutilizar el índice ya creado.
RECREATE_INDEX = True

if RECREATE_INDEX and vector_store.client.indices.exists(index=INDEX_NAME):
    vector_store.client.indices.delete(index=INDEX_NAME)
    print(f"Índice eliminado: {INDEX_NAME}")

ids = [doc.metadata["chunk_id"] for doc in splits]

inserted_ids = vector_store.add_documents(
    documents=splits,
    ids=ids,
    bulk_kwargs={"chunk_size": 100},
)

vector_store.client.indices.refresh(index=INDEX_NAME)
print(f"Chunks indexados: {len(inserted_ids)}")

Índice eliminado: rag_hiraoka_terminosycondiciones
Chunks indexados: 113


## 1.7 Validación rápida de recuperación vectorial

Probamos una búsqueda semántica simple antes de construir el agente.

In [53]:
from collections import Counter


def score_label(score):

    if score is None:
        return ""

    if score >= 0.90:
        return "🟢 Excelente"

    if score >= 0.80:
        return "🟡 Bueno"

    return "🔴 Débil"


def print_retrieval_results(results, max_chars=600):

    print("\n" + "=" * 120)
    print("RESULTADOS DE RECUPERACIÓN")
    print("=" * 120)

    section_counter = Counter()

    for i, item in enumerate(results, start=1):

        # Compatible con similarity_search()
        # y similarity_search_with_score()

        if isinstance(item, tuple):
            doc, score = item
        else:
            doc, score = item, None

        meta = doc.metadata

        section_slug = meta.get("section_slug", "N/A")
        section_counter[section_slug] += 1

        print("\n" + "=" * 120)

        header = f"Resultado {i}"

        if score is not None:
            header += (
                f" | score={score:.4f}"
                f" ({score_label(score)})"
            )

        print(header)

        print("-" * 120)

        print(
            f"Chunk ID      : "
            f"{meta.get('chunk_id')}"
        )

        print(
            f"Documento     : "
            f"{meta.get('documento')}"
        )

        print(
            f"Sección       : "
            f"{meta.get('section_title')} "
            f"({meta.get('section_slug')})"
        )

        print(
            f"Páginas       : "
            f"{meta.get('page_start')} "
            f"- "
            f"{meta.get('page_end')}"
        )

        print(
            f"Autor         : "
            f"{meta.get('autor')}"
        )

        print(
            f"Año           : "
            f"{meta.get('anio')}"
        )

        print(
            f"Dominio       : "
            f"{meta.get('dominio')}"
        )

        print(
            f"Categoría SAC : "
            f"{meta.get('sac_category', 'N/A')}"
        )

        print(
            f"Tipo Cláusula : "
            f"{meta.get('clause_type', 'N/A')}"
        )

        print(
            f"Nivel Riesgo  : "
            f"{meta.get('risk_level', 'N/A')}"
        )

        entities = meta.get("entities", [])

        print(
            f"Entidades     : "
            f"{', '.join(entities) if entities else 'N/A'}"
        )

        print("-" * 120)

        preview = (
            doc.page_content[:max_chars]
            .replace("\n", " ")
            .strip()
        )

        print(preview)

    print("\n" + "=" * 120)
    print("DISTRIBUCIÓN POR SECCIONES")
    print("=" * 120)

    for section, qty in section_counter.items():
        print(f"{section:<30} {qty}")

In [54]:
query = "Quería solicitar que cambien el distrito a LURIGANCHO - CHOSICA para el delivery"

results = vector_store.similarity_search_with_score(
    query,
    k=5
)

print_retrieval_results(results)


RESULTADOS DE RECUPERACIÓN

Resultado 1 | score=0.8653 (🟡 Bueno)
------------------------------------------------------------------------------------------------------------------------
Chunk ID      : delivery_tipo_0085
Documento     : Terminos y Condiciones
Sección       : Mecanismos de entrega de productos (entrega_productos)
Páginas       : 11 - 13
Autor         : Hiraoka
Año           : 2026
Dominio       : Términos y Condiciones
Categoría SAC : N/A
Tipo Cláusula : N/A
Nivel Riesgo  : N/A
Entidades     : N/A
------------------------------------------------------------------------------------------------------------------------
2) Tipo de servicio de envío (Delivery) elegido:   Tipo de servicio Detalle Envío Hoy   La entrega del producto se realizará observando lo siguiente: Si la compra de los productos/pedido se realizan entre las 07:00 y 13:00 horas, recibirá el producto durante el día que realizó la compra desde las 15:00 horas hasta las 20:00 horas. Los productos/pedidos que 

# Parte 2 — Agente básico con RAG mediante herramienta propia

En esta parte construiremos un agente con `create_agent`, pero la recuperación no se expondrá con `retriever.as_tool()` ni con `chain.as_tool()`.

Crearemos una herramienta propia con `@tool` que envuelve el `vector_store` y permite filtros por metadata. Este enfoque permite controlar:

- `section_slug`
- rango de páginas
- `tipo_contenido`
- `temas`
- cantidad de resultados
- formato de salida con fuentes

## 2.1 Constructor de filtros por metadata

Elasticsearch permite filtrar usando expresiones como `term`, `terms` y `range`. La metadata quedó guardada dentro del objeto `metadata`, por eso los filtros usan campos como `metadata.section_slug.keyword` o `metadata.page_number`.

In [55]:
from typing import Optional


def build_metadata_filter(
    section_slug: Optional[str] = None,
    documento: Optional[str] = None,
) -> list[dict]:
    """
    Construye filtros Elasticsearch para búsqueda híbrida RAG.

    Metadata soportada actualmente:
    - documento
    - section_slug (soporta buscar en cualquier nivel de la jerarquía: section, subsection, sub_subsection)
    """

    filters = []

    if documento:
        filters.append({
            "term": {
                "metadata.documento.keyword": documento
            }
        })

    if section_slug:
        filters.append({
            "bool": {
                "should": [
                    {"term": {"metadata.section_slug.keyword": section_slug}},
                    {"term": {"metadata.subsection_slug.keyword": section_slug}},
                    {"term": {"metadata.sub_subsection_slug.keyword": section_slug}}
                ],
                "minimum_should_match": 1
            }
        })

    return filters

In [56]:
filters = build_metadata_filter(
    documento="Terminos y Condiciones",
    section_slug="delivery"
)

print(filters)

[{'term': {'metadata.documento.keyword': 'Terminos y Condiciones'}}, {'bool': {'should': [{'term': {'metadata.section_slug.keyword': 'delivery'}}, {'term': {'metadata.subsection_slug.keyword': 'delivery'}}, {'term': {'metadata.sub_subsection_slug.keyword': 'delivery'}}], 'minimum_should_match': 1}}]


## 2.2 Herramienta RAG personalizada

La herramienta recibe parámetros semánticos y estructurados. Internamente construye filtros de Elasticsearch y ejecuta la búsqueda sobre el vector store.

In [58]:
from typing import Optional

from pydantic import BaseModel, Field


class BuscarTerminosHiraokaInput(BaseModel):

    consulta: str = Field(
        description="Pregunta relacionada con los términos y condiciones de Hiraoka."
    )

    k: int = Field(
        default=5,
        ge=1,
        le=10,
        description="Cantidad máxima de fragmentos a recuperar."
    )

    section_slug: Optional[str] = Field(
        default=None,
        description="""
        Sección específica del documento.

        Ejemplos:
        - delivery
        - recojo_tienda
        - garantias
        - metodos_pago
        - contacto
        - precio_stock
        """
    )

In [59]:
SECTION_HINTS = {
    # Delivery
    "delivery": "delivery",
    "envio": "delivery",
    "envío": "delivery",
    "despacho": "delivery",
    "entrega": "delivery",
    "costo": "delivery_costos",
    "tarifa": "delivery_costos",
    "precio envio": "delivery_costos",
    "precio envío": "delivery_costos",

    # Recojo en tienda
    "recojo": "recojo_tienda",
    "retiro": "recojo_tienda",
    "recoger": "recojo_tienda",

    # Oka
    "oka": "pago_oka",
    "cuotas": "pago_oka",
    "credito": "pago_oka",
    "crédito": "pago_oka",

    # Izipay & PagoEfectivo
    "izipay": "pago_izipay",
    "pagoefectivo": "pago_efectivo",

    # Horarios / Tiendas / Contacto
    "horario": "contacto",
    "telefono": "contacto",
    "teléfono": "contacto",
    "correo": "contacto",
    "contactar": "contacto",

    # Garantías & Devoluciones
    "garantia": "garantias",
    "garantía": "garantias",

    # Combo
    "combo": "arma_combo",
}

In [60]:
def format_results_for_agent(
    results,
    max_chars: int = 1200
):

    if not results:
        return (
            "No se encontró información relevante "
            "en los Términos y Condiciones."
        )

    blocks = []

    for i, (doc, score) in enumerate(results, start=1):

        meta = doc.metadata

        content = (
            doc.page_content[:max_chars]
            .replace("\n", " ")
            .strip()
        )

        # Construir título jerárquico de la sección
        sec_title = meta.get('section_title', '')
        if meta.get('subsection_title'):
            sec_title += f" -> {meta.get('subsection_title')}"
        if meta.get('sub_subsection_title'):
            sec_title += f" -> {meta.get('sub_subsection_title')}"

        blocks.append(
            f"""
[Fuente {i}]

Documento: {meta.get('documento')}

Sección: {sec_title}

Slug: {meta.get('sub_subsection_slug') or meta.get('subsection_slug') or meta.get('section_slug')}

Páginas: {meta.get('page_start')} - {meta.get('page_end')}

Chunk ID: {meta.get('chunk_id')}

Score: {score:.4f}

Contenido:
{content}
"""
        )

    return "\n\n".join(blocks)

### Herramienta principal

In [61]:
from langchain.tools import tool


@tool(args_schema=BuscarTerminosHiraokaInput)
def buscar_terminos_hiraoka(
    consulta: str,
    k: int = 5,
    section_slug: Optional[str] = None,
) -> str:
    """
    Busca información dentro de los
    Términos y Condiciones de Hiraoka
    utilizando Elasticsearch.
    """

    # Detección automática de sección

    if not section_slug:

        query_lower = consulta.lower()

        for keyword, detected_section in SECTION_HINTS.items():

            if keyword in query_lower:

                section_slug = detected_section
                break

    filters = build_metadata_filter(
        section_slug=section_slug,
        documento="Terminos y Condiciones"
    )

    results = vector_store.similarity_search_with_score(
        query=consulta,
        k=k,
        filter=filters or None,
    )

    results = [
        r for r in results
        if r[1] >= 0.65
    ]

    results = sorted(
        results,
        key=lambda x: x[1],
        reverse=True
    )

    return format_results_for_agent(results)

In [63]:
print(
    buscar_terminos_hiraoka.invoke(
        {
            "consulta": "Quiero devolverlo o cambiar por otro celular"
        }
    )
)


[Fuente 1]

Documento: Terminos y Condiciones

Sección: Enlaces (Links)

Slug: enlaces

Páginas: 16 - 17

Chunk ID: enlaces_0109

Score: 0.8225

Contenido:
CDPC. Hiraoka deja expresa constancia que la restitución o devolución de los productos digitales, de software, o que involucren licencias (por ejemplo: programas) no será aplicable debido a la naturaleza de los mismos (no reutilizables). En ese sentido, no procederá la soli



[Fuente 2]

Documento: Terminos y Condiciones

Sección: Procedimiento de Selección de Productos y Pago -> Medidas de seguridad adoptadas en el procedimiento de pago -> PagoEfectivo

Slug: pago_efectivo

Páginas: 5 - 5

Chunk ID: pago_efectivo_0040

Score: 0.8184

Contenido:
maneja una transferencia (hasta 15 días hábiles posteriores al procesamiento de la solicitud) a cuenta de titular de la compra. En caso de solicitud de devoluciones de pagos a través de Izipay se deberá de enviar un correo a Servicio Cliente Hiraoka a través de: servicioalcliente@hiraoka.c

In [29]:
print(splits[0].metadata)

{'documento': 'Terminos y Condiciones', 'autor': 'Hiraoka', 'anio': 2026, 'dominio': 'Términos y Condiciones', 'page_start': 1, 'page_end': 1, 'section_slug': 'procedimiento_pago', 'section_title': 'Procedimiento de Selección de Productos y Pago', 'subsection_slug': 'seguridad_pago', 'subsection_title': 'Medidas de seguridad adoptadas en el procedimiento de pago', 'sub_subsection_slug': 'pago_oka', 'sub_subsection_title': 'Oka', 'chunk_id': 'pago_oka_0000', 'chunk_number': 0}


## 2.3 Prueba directa de la herramienta

Antes de conectar la herramienta al agente, conviene probarla de forma aislada.

In [64]:
test_questions = [

    "¿Cuál es el horario de atención los sábados?",

    "¿Cuántos días tengo para recoger mi producto?",

    "¿Qué pasa si no recojo mi pedido?",

    "¿Cuánto demora el delivery?",

    "¿Qué cubre la garantía?",

    "¿Puedo devolver un producto?",

    "¿Qué métodos de pago aceptan?"
]

for question in test_questions:

    print("\n\n")
    print("#" * 120)
    print("PREGUNTA")
    print(question)
    print("#" * 120)

    results = vector_store.similarity_search_with_score(
        question,
        k=3
    )

    print_retrieval_results(
        results,
        max_chars=400
    )




########################################################################################################################
PREGUNTA
¿Cuál es el horario de atención los sábados?
########################################################################################################################

RESULTADOS DE RECUPERACIÓN

Resultado 1 | score=0.8096 (🟡 Bueno)
------------------------------------------------------------------------------------------------------------------------
Chunk ID      : horario_atencion_0068
Documento     : Terminos y Condiciones
Sección       : Mecanismos de entrega de productos (entrega_productos)
Páginas       : 9 - 10
Autor         : Hiraoka
Año           : 2026
Dominio       : Términos y Condiciones
Categoría SAC : N/A
Tipo Cláusula : N/A
Nivel Riesgo  : N/A
Entidades     : N/A
------------------------------------------------------------------------------------------------------------------------
con la Av. El Bosque) Lunes - Sábado: 10:00 a.m. a  8:00 p

## 2.4 Crear agente básico con LangChain 1.x

El agente tendrá una sola herramienta de recuperación. La instrucción principal obliga a responder con sustento: documento, sección y página.

In [65]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    CHAT_MODEL,
    temperature=0
)

SYSTEM_PROMPT = """
Eres un asistente especializado en los Términos y Condiciones de Hiraoka.

Tu objetivo es responder consultas relacionadas con compras, delivery,
recojo, pagos, garantías, devoluciones y demás condiciones comerciales
de Hiraoka.

INFORMACIÓN FIJA (NO ES NECESARIO CONSULTAR LA HERRAMIENTA PARA ESTOS CASOS)

Datos de contacto:
- Teléfono: (01) 680-3800
- WhatsApp: 969872372
- Correo: servicioalcliente@hiraoka.com.pe

Información de la empresa:
- Razón social: IMPORTACIONES HIRAOKA S.A.C.
- RUC: 20100016681
- Dirección: Av. Abancay 594, Cercado de Lima

Métodos de pago disponibles:
- Izipay
- PagoEfectivo
- OKA

Información sobre OKA:
- Los créditos están sujetos a evaluación.
- OKA es un proveedor independiente de servicios financieros.
- Para consultas de crédito, comunicarse al (01) 705-1717 con OKA.
- Para devoluciones de compras realizadas con OKA se procede con la anulación del crédito.

REGLAS DE USO DE LA HERRAMIENTA

Utiliza la herramienta buscar_terminos_hiraoka cuando la consulta requiera
información específica que no se encuentre en la información fija anterior.

Especialmente para temas relacionados con:

- Delivery
- Cobertura de distritos
- Costos de envío
- Envío Hoy
- Reprogramaciones
- Recojo en tienda
- Garantías
- Devoluciones
- Cambios
- Promociones
- Stock
- Restricciones
- Plazos
- Penalidades
- Validación de compras
- Cualquier cláusula específica de los Términos y Condiciones

INSTRUCCIONES

1. No inventes información.

2. Si la herramienta no devuelve evidencia suficiente, responde:
   "No encontré información suficiente en los Términos y Condiciones de Hiraoka para responder esta consulta."

3. Prioriza la información más específica y relevante recuperada.

4. Si existen varias condiciones aplicables, explica las diferencias.

5. Responde siempre en español.

6. Si la consulta está fuera del alcance de los Términos y Condiciones,
   indícalo claramente.

7. Si la consulta es únicamente sobre:
   - teléfono
   - whatsapp
   - correo
   - dirección
   - métodos de pago
   - información general de OKA

   NO utilices la herramienta.
8. Si la consulta es sobre horarios de atención, horarios de tienda:
    - El horario de atención en la tienda Hiraoka Lima es de Lunes a Viernes de 10:00 a.m. a 8:00 p.m. , los Sábados de 10:00 a.m. a 9:00 p.m. y los Domingos desde las 10:30 a.m. a 7:00 p.m.
    - El horario de atención en la tienda Hiraoka Miraflores es de Lunes a Viernes de 10:00 a.m. a 8:00 p.m. , los Sábados de 10:00 a.m. a 9:00 p.m. y los Domingos desde las 10:30 a.m. a 7:00 p.m.
    - El horario de atención en la tienda Hiraoka San Miguel es de Lunes a Viernes de 10:00 a.m. a 8:00 p.m. , los Sábados de 10:00 a.m. a 9:00 p.m. y los Domingos desde las 10:30 a.m. a 7:00 p.m.
    - El horario de atención en la tienda Hiraoka Independencia es de Lunes a Viernes de 10:00 a.m. a 8:00 p.m. , los Sábados de 10:00 a.m. a 9:00 p.m. y los Domingos desde las 10:30 a.m. a 7:00 p.m.
    - El horario de atención en la tienda Hiraoka San Juan de Lurigancho es de Lunes a Viernes de 10:00 a.m. a 8:00 p.m. , los Sábados de 10:00 a.m. a 9:00 p.m. y los Domingos desde las 10:30 a.m. a 7:00 p.m.

    Este horario aplica para los locales de Lima, Miraflores, San Miguel, San Juan de Lurigancho e Independencia.
    Para el sitio web esta disponible las 24 horas.

9. Si la consulta menciona:
   - garantía
   - devolución
   - delivery
   - envío
   - cobertura
   - recojo
   - stock
   - promoción
   - validación
   - reprogramación

   utiliza la herramienta para obtener evidencia.

FORMATO DE CITAS

Cuando uses información recuperada desde la herramienta, agrega:

Fuentes:
- Documento: <documento>
- Sección: <section_title>
- Páginas: <page_start>-<page_end>

Si la respuesta proviene únicamente de la información fija, no es necesario mostrar fuentes.
"""

agent = create_agent(
    model=model,
    tools=[buscar_terminos_hiraoka],
    system_prompt=SYSTEM_PROMPT,
)

In [69]:
question = "Si el iPhone qué les he comprado tiene problemas de sistema uds lo ven o tengo que llevarlo a la tienda de iPhone?"
response = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'No encontré información suficiente en los Términos y Condiciones de Hiraoka para responder esta consulta. La sección de garantías menciona que Hiraoka se rige por su "Política de Cambios y Devoluciones y Garantías", pero no detalla el procedimiento específico para problemas de sistema en productos como un iPhone ni si Hiraoka realiza el servicio técnico directamente o si se debe acudir a un centro autorizado de la marca.', 'extras': {'signature': 'Cu8FAQw51sfI08xLqrkrlGW46h5N6TpjORMRtR6W7WQyT1I9DE0Dzt1oItoU5M82QIbF8Fw/gERmINBkaHByNHnAI3rnHFxIuHbgf+RWcVrKHygruvneEwinmLw5L3xa9oikhiretrGtbFdiarYORs3lQWPUSIhyE181VTOwsXZg5XoiU+EL0HxSi3pzk+E5l1K/wBjoZm9xBNm9yrWI9CW+SC/79HdKTRSZMT3PLGynu74e8j7AoEDRVE+PwLjjxUku48n0WbylXcmCY76D+zYjq1MuiaVSiA6NrBVS7GxHczLP2zlJpyVc6NWFIOGBMPQxpmirXKsCKGqAfkKfzXNBewrePzgcGkORl5xL0hNMcGRXo3JR1RM67RzILyFhsyR5FLBi/W4piAH0eyFueEDAAfseAXvkBMNAWhp7vBom6tXpEPFnikIatmtY7FtO3FX9p3TyGeIxf1tZv4QXTA403qwPj4HKL80Hdryto9wGuJVSMo1y39U82C8Av/YUttwrGA+i/

## 2.5 Preguntas de práctica para el agente

In [70]:
preguntas_practica = [

    "Si el iPhone qué les he comprado tiene problemas de sistema uds lo ven o tengo que llevarlo a la tienda de iPhone?",

    "¿Cuántos días tengo para recoger un producto comprado por la web?",

    "¿Qué sucede si no recojo mi pedido dentro del plazo establecido?",

    "¿Cómo funciona el servicio de delivery de Hiraoka?",

    "¿Qué distritos están cubiertos por el servicio de entrega?",

    "¿Qué métodos de pago acepta Hiraoka?",

    "¿Cómo funcionan las compras mediante OKA?",

    "¿Qué condiciones aplican a las promociones Arma tu Combo?",

    "¿Qué información brinda el documento sobre garantías de productos?",

    "¿Puede variar el stock mostrado en la página web?",

    "¿Qué ocurre si un producto comprado no se encuentra disponible?",

    "¿Cómo puedo contactar a Hiraoka para consultas o reclamos?",

    "¿Qué situaciones se consideran casos de fuerza mayor?",

    "¿Cómo funciona la reprogramación de entregas?",

    "¿Qué restricciones existen para el recojo por terceros?"
]

In [71]:
for pregunta in preguntas_practica[:5]:

    print("=" * 120)
    print("PREGUNTA:")
    print(pregunta)

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": pregunta
                }
            ]
        }
    )

    print("\nRESPUESTA:")
    print(response["messages"][-1].content)
    print("\n")

PREGUNTA:
Si el iPhone qué les he comprado tiene problemas de sistema uds lo ven o tengo que llevarlo a la tienda de iPhone?

RESPUESTA:
[{'type': 'text', 'text': 'No encontré información suficiente en los Términos y Condiciones de Hiraoka para responder esta consulta. La sección de garantías menciona que Hiraoka se rige por su "Política de Cambios y Devoluciones y Garantías", pero no detalla el procedimiento específico para problemas de sistema en productos como un iPhone ni si la atención la brinda Hiraoka directamente o un servicio técnico autorizado de la marca.', 'extras': {'signature': 'CswFAQw51sejJUO6fc76HhxbUxw8vvsHk8S6iRhTJZEC8y+nu3eNMD/gE/j2yb+S01L5cOD8h7kChDAg2gPCLgz5IKQLuVd0amaVVOcrox7jIOP94oZd2d8cpjsEsL9bvFpZ9aFKyMORfB46RDfB9qcLg1zD8Op3DRtzYV6e5I6zXbiDkVnDLRNpeMCLLkhz5AASyWadq+zCA2PNPxwpNL/cqYqBygdXnZm/K7gvfdEppqUo4fij4QDuo5Op2IJVnCMhUJC3ftdnHrKS5MQ8Iq2G2dyOEIoib8iJy4S/CGlSRa2hnaAbJwNiEw47NOIsLMyF1V1jGKHNJgF4TKRuOvRXcptUCuSP7hm1BxdhucZ8rpUPlZ1tCYcWJa3kGwYAisDJmo/sw01R05BX

KeyboardInterrupt: 

In [35]:
results = vector_store.similarity_search_with_score(
    "Quiero devolverlo o cambiar por otro celular",
    k=10
)

print_retrieval_results(results)


RESULTADOS DE RECUPERACIÓN

Resultado 1 | score=0.8201 (🟡 Bueno)
------------------------------------------------------------------------------------------------------------------------
Chunk ID      : reprogramaciones_0080
Documento     : Terminos y Condiciones
Sección       : Reprogramaciones (reprogramaciones)
Páginas       : 15 - 16
Autor         : Hiraoka
Año           : 2026
Dominio       : Términos y Condiciones
Categoría SAC : N/A
Tipo Cláusula : N/A
Nivel Riesgo  : N/A
Entidades     : N/A
------------------------------------------------------------------------------------------------------------------------
entrega de productos no dará lugar a compensación económica, reembolso o responsabilidad adicional por parte de Hiraoka cuando dichos cambios se generen bajo los supuestos previamente mencionados y hayan sido previa y debidamente informados por Hiraoka. Asimismo, los Usuarios aceptan que es su responsabilidad gestionar cualquier solicitud por inconveniente derivado de la r

In [36]:
print(
    buscar_terminos_hiraoka.invoke(
        {
            "consulta":"Quiero devolverlo o cambiar por otro celular"
        }
    )
)


[Fuente 1]

Documento: Terminos y Condiciones

Sección: Reprogramaciones

Slug: reprogramaciones

Páginas: 15 - 16

Chunk ID: reprogramaciones_0080

Score: 0.8201

Contenido:
entrega de productos no dará lugar a compensación económica, reembolso o responsabilidad adicional por parte de Hiraoka cuando dichos cambios se generen bajo los supuestos previamente mencionados y hayan sido previa y debidamente informados por Hiraoka. Asimismo, los Usuarios aceptan que es su responsabilidad gestionar cualquier solicitud por inconveniente derivado de la reprogramación de entregas. Para ello, el Usuario cuenta con un plazo de hasta siete (7) días naturales a partir de la fecha de notificada la reprogramación de su pedido y deberá de contactarse al número: (01) 680-3800



[Fuente 2]

Documento: Terminos y Condiciones

Sección: Procedimiento de Selección de Productos y Pago -> Medidas de seguridad adoptadas en el procedimiento de pago -> PagoEfectivo

Slug: pago_efectivo

Páginas: 5 - 5

Chunk ID:

# -----------------------